# 01 · Ympäristö ja testit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/markomanninen/tgd-geometry-lab/blob/main/notebooks/01_ymparisto_ja_testit.ipynb)

Tämä kirja on rakennettu Lanyonin Colab-esimerkin (*Run LanyonScripts in Colab*) mallin mukaan:
väliaikainen Ubuntu-ajoympäristö, GNU Maxima, Prodanovin `clifford`/`cliffordan` -paketit ja
skriptien ajo omissa Maxima-prosesseissaan.

Erona Lanyonin esimerkkiin: jokainen moduuli tarkistetaan **automaattisilla testeillä**
(tunnetut identiteetit kuten $d^2=0$, $d\omega=0$, $\mathrm{Ric}=3g$, Bianchi), jotta
laskentakirjaston virheet eivät sekoitu TGD:n fysikaalisiin oletuksiin (välitavoite M1).

In [1]:
# Asennus: Maxima, tämä repositorio ja Dimiter Prodanovin clifford-paketti.
# Colabissa ajo kestää noin minuutin; paikallisesti asetetaan TGD_ROOT / CLIFFORD_PATH.
import os, sys, shutil
ROOT = os.environ.get("TGD_ROOT", "/content/tgd-geometry-lab")
CLI  = os.environ.get("CLIFFORD_PATH", "/content/clifford")
if not shutil.which("maxima"):
    !apt-get -qq update > /dev/null && apt-get -qq install -y maxima > /dev/null
if not os.path.exists(ROOT):
    !git clone -q --depth 1 https://github.com/markomanninen/tgd-geometry-lab.git {ROOT}
if not os.path.exists(CLI):
    !git clone -q --depth 1 https://github.com/dprodanov/clifford.git {CLI}
sys.path.insert(0, ROOT + "/notebooks")
from tgdlab import Lab
lab = Lab(ROOT, CLI)
!maxima --version 2>/dev/null | tail -1

Maxima 5.46.0


## Testisarja

Ajaa kaikki `tests/`-hakemiston testit (metriikka, kaarevuus, muodot, $CP_2$, upotus, toiminta, Clifford).

In [2]:
ok = lab.tests()
print('KAIKKI TESTIT LÄPI' if ok else 'TESTEJÄ EPÄONNISTUI')

--- core/metric --- 
  PASS  inverse metric of S^2 
  PASS  determinant of S^2 metric 
  PASS  Minkowski signature (1,3) 
  PASS  direct sum dimension 
  PASS  tensor transformation to polar coordinates 
--- geometry/curvature (unit S^2 reference) --- 
  PASS  S^2 Christoffel Gamma^th_{ph ph} = -sin cos 
  PASS  S^2 Ricci = g 
  PASS  S^2 scalar curvature = 2 
  PASS  S^2 metric compatibility nabla g = 0 
--- forms --- 
  PASS  dx1^dx2 = -dx2^dx1 
  PASS  dx1^dx1 = 0 
  PASS  d^2 = 0 on a generic 0-form 
  PASS  d^2 = 0 on a generic 1-form 
  PASS  graded Leibniz rule 
  PASS  matrix <-> 2-form round trip 
  PASS  Euclidean R^4: *dx1 = dx2^dx3^dx4 
  PASS  Euclidean R^4: ** = +1 on 2-forms 
  PASS  Minkowski: ** = -1 on 2-forms 
--- spaces/cp2 (Fubini-Study) --- 
  PASS  metric symmetric 
  PASS  det g = 16/(1+rho^2)^6 
  PASS  metric positive definite at a sample point 
  PASS  inverse metric 
  PASS  J^2 = -1 
  PASS  J is g-orthogonal: J^T g J = g 
  PASS  d omega = 0 (Kahler) 
  PA

## Clifford-algebra (Prodanov)

Sama paketti kuin LanyonScriptsissä. Tarkistetaan Clifford-relaatiot
$\gamma_a\gamma_b+\gamma_b\gamma_a = 2\eta_{ab}$ signatuurilla $(1,3)$.

In [3]:
print(lab.values("clifford(g, 1, 3)$",
    ["g[1] . g[1]", "g[2] . g[2]", "g[1] . g[2] + g[2] . g[1]", "g[3] . g[4] + g[4] . g[3]"],
    load_clifford=True))

['1', '-1', '0', '0']


## Vertailu: LanyonScriptsin esimerkit

Valinnainen: ajetaan Lanyonin kuusi virtausfysiikan esimerkkiä (Burgers, matala vesi, Euler, Maxwell, MHD)
samalla tavalla kuin alkuperäisessä kirjassa. Ne käyttävät samaa Maxima–Clifford-pohjaa,
jonka päälle TGD Geometry Lab rakentaa differentiaaligeometrian.

In [4]:
RUN_LANYON = os.environ.get("RUN_LANYON", "1") == "1"   # aseta 0 ohittaaksesi
if RUN_LANYON:
    import subprocess
    from pathlib import Path
    L = Path("/content/LanyonScripts")
    if not L.exists():
        !git clone -q --depth 1 https://github.com/lanyonai/LanyonScripts.git {L}
    for f in ["burgers.mac", "shallow-water.mac", "isoeuler.mac", "euler.mac", "maxwell.mac", "ideal-mhd.mac"]:
        code = (f'file_search_maxima: append(file_search_maxima, ["{CLI}/$$$.{{mac,wxm}}"])$ '
                f'load("{L}/eqnsys/{f}")$ print("LANYON_SCRIPT_FINISHED")$')
        p = subprocess.run(["maxima", "--very-quiet", f"--batch-string={code}"],
                           capture_output=True, text=True, timeout=300)
        print(f"{f:20s}", "OK" if "LANYON_SCRIPT_FINISHED" in p.stdout else "VIRHE")